In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load dataset
sales = pd.read_csv("../data/processed/daily_sales.csv")


In [3]:
# Convert date
sales["date"] = pd.to_datetime(sales["date"])

In [4]:
# -----------------------------
# WEEKLY AGGREGATION
# -----------------------------

weekly_sales = sales.resample(
    "W",
    on="date"
)["sales"].sum().reset_index()

In [5]:

# -----------------------------
# TIME FEATURES
# -----------------------------

In [6]:
weekly_sales["year"] = (
    weekly_sales["date"].dt.year
)

In [7]:
weekly_sales["month"] = (
    weekly_sales["date"].dt.month
)

In [8]:
weekly_sales["quarter"] = (
    weekly_sales["date"].dt.quarter
)

In [9]:
weekly_sales["week_of_year"] = (
    weekly_sales["date"]
    .dt.isocalendar()
    .week
)

In [10]:
weekly_sales["is_month_start"] = (
    weekly_sales["date"]
    .dt.is_month_start.astype(int)
)

In [11]:
weekly_sales["is_month_end"] = (
    weekly_sales["date"]
    .dt.is_month_end.astype(int)
)

In [12]:
# -----------------------------
# LAG FEATURES
# -----------------------------

In [13]:
weekly_sales["lag_1"] = (
    weekly_sales["sales"].shift(1)
)

weekly_sales["lag_4"] = (
    weekly_sales["sales"].shift(4)
)

weekly_sales["lag_8"] = (
    weekly_sales["sales"].shift(8)
)

In [14]:
# -----------------------------
# ROLLING FEATURES
# -----------------------------

In [15]:
weekly_sales["rolling_mean_4"] = (
    weekly_sales["sales"]
    .rolling(window=4)
    .mean()
)

In [16]:
weekly_sales["rolling_mean_8"] = (
    weekly_sales["sales"]
    .rolling(window=8)
    .mean()
)

In [17]:
weekly_sales["rolling_std_4"] = (
    weekly_sales["sales"]
    .rolling(window=4)
    .std()
)


In [18]:
weekly_sales["ema_4"] = (
    weekly_sales["sales"]
    .ewm(span=4)
    .mean()
)

In [19]:
# Trend feature
weekly_sales["trend"] = range(
    len(weekly_sales)
)


In [20]:
# -----------------------------
# OUTLIER REMOVAL
# -----------------------------


In [21]:
q1 = weekly_sales["sales"].quantile(0.25)
q3 = weekly_sales["sales"].quantile(0.75)

iqr = q3 - q1

weekly_sales = weekly_sales[
    ~(
        (weekly_sales["sales"] < (q1 - 1.5 * iqr)) |
        (weekly_sales["sales"] > (q3 + 1.5 * iqr))
    )
]

In [22]:
# -----------------------------
# REMOVE NULLS
# -----------------------------

weekly_sales.dropna(inplace=True)

In [24]:
# -----------------------------
# SAVE FEATURE DATASET
# -----------------------------

weekly_sales.to_csv(
    "../data/processed/feature_engineered_sales.csv",
    index=False
)

In [25]:
weekly_sales.head()

,date,sales,year,month,quarter,week_of_year,is_month_start,is_month_end,lag_1,lag_4,lag_8,rolling_mean_4,rolling_mean_8,rolling_std_4,ema_4,trend
8,2016-11-13,0,2016,11,4,45,0,0,0.0,37.0,3.0,0.0,39.125,0.0,10.630066,8
9,2016-11-20,0,2016,11,4,46,0,0,0.0,0.0,0.0,0.0,39.125,0.0,6.352173,9
10,2016-11-27,0,2016,11,4,47,0,0,0.0,0.0,0.0,0.0,39.125,0.0,3.802052,10
11,2016-12-04,0,2016,12,4,48,0,0,0.0,0.0,276.0,0.0,4.625,0.0,2.277913,11
12,2016-12-11,0,2016,12,4,49,0,0,0.0,0.0,37.0,0.0,0.000,0.0,1.365556,12
